# OpenAI Finetune API

## Importing the libraries

In [ ]:
import json
from openai import OpenAI

In [ ]:
client = OpenAI()

## Creating the JSONL input files

In [ ]:
questions = [
    "What is Python?",
    "What is JavaScript?",
    "What is an API?"
]

answers = [
    "Python is a high-level, general-purpose programming language.",
    "JavaScript is a programming language commonly used to create interactive web applications.",
    "An API is an interface that allows different software applications to communicate with each other."
]

with open("train.jsonl", "w", encoding="utf-8") as f:
    for question, answer in zip(questions, answers):
        example = {
            "messages": [
                {
                    "role": "user",
                    "content": question
                },
                {
                    "role": "assistant",
                    "content": answer
                }
            ]
        }

        f.write(json.dumps(example) + "\n")

print("Created train.jsonl!")

In [ ]:
questions = [
    "What is Python commonly used for?",
    "What role does JavaScript play in web development?",
    "Why are APIs useful in software applications?"
]

answers = [
    "Python is commonly used for web development, automation, data analysis, machine learning, and scripting.",
    "JavaScript is used in web development to add interactive and dynamic behavior to web pages.",
    "APIs are useful because they allow different software applications and services to communicate and exchange data."
]

with open("val.jsonl", "w", encoding="utf-8") as f:
    for question, answer in zip(questions, answers):
        example = {
            "messages": [
                {
                    "role": "user",
                    "content": question
                },
                {
                    "role": "assistant",
                    "content": answer
                }
            ]
        }

        f.write(json.dumps(example) + "\n")

print("Created val.jsonl!")

## Uploading the JSONL files to OpenAI

In [ ]:
training_file = client.files.create(
    file = open("train.jsonl", "rb"),
    purpose = "fine-tune"
)

print("Input file ID: ", training_file.id)

In [ ]:
validation_file = client.files.create(
    file = open("val.jsonl", "rb"),
    purpose = "fine-tune"
)

print("Input file ID: ", validation_file.id)

## Creating the finetune jobs

In [ ]:
finetune = client.fine_tuning.jobs.create(
    training_file = training_file.id,
    validation_file = validation_file.id,
    model = "gpt-4.1-mini",
    hyperparameters = {
        "n_epochs": 1,
        "batch_size": 1
    },
    suffix = "My-Finetune"
)

print("Job ID: ", finetune.id)
print("Job Details: ", client.fine_tuning.jobs.list(limit = 1))

## Checking the finetune status

In [ ]:
finetune = client.fine_tuning.jobs.retrieve(finetune.id)

print("Job ID: ", finetune.id)
print("List of events: ", finetune.list_events(finetune_job_id = finetune.id, limit = 10).data)
print("Status: ", finetune.status)
print("Output file ID: ", finetune.output_file_id)
print("Error file ID: ", finetune.error_file_id)

Typical statuses include `validating`, `in_progress`, `finalizing`, and `completed`.

## Retreiving the finetuned model

In [ ]:
finetune = client.fine_tuning.jobs.retrieve(finetune.id)

if finetune.status != "completed":
    print(f"Finetune is not completed yet. Current status: {finetune.status}")
elif not finetune.output_file_id:
    print("Finetune completed but no model returned.")
else:
    finetuned = finetune.fine_tuned_model

## Calling the finetuned model

In [ ]:
message = "Hey ChatGPT!, Find x when x + 10 = 20x?"

response = client.responses.create(
    model = finetuned,
    input = [
        {"role": "system", "content": "Act like a math teacher"},
        {"role": "user", "content": message}  
    ]
)

print(response.output_text)